# Bruise Distillation Study — full pipeline on ORC (one GPU, resumable)

Runs the **whole distillation study** you set up on the server, on a single ORC GPU:
validation oracle → Phase-2 baselines (response, CWD, BPKD, uniform ensemble) →
Phase-3 novel method (reliability-gated B2+B5) + ablations (+boundary/+hard/+group/+full) →
extras (angular, DKD) → the final comparison table.

### Key safety property (what you asked for)
Every arm **resumes** from `resume_checkpoint.pt` (written every epoch) and only writes
`DONE.json` + its test scores **after training fully completes** (all epochs OR early stop).
So a 12 h wall-time kill loses at most one epoch, and an **interrupted arm is never treated
as finished** — re-running the loop cell skips completed arms and continues the interrupted one.

### Before you run
1. Copy **`orc_bundle.zip`** (built by `make_orc_bundle.py` on the server) to `/scratch/$USER/`.
   It contains the trained teachers (best B5 seed, B2, B0-distilled), encoders, manifests,
   ITA labels, test/val images, the reference scores, and the code.
2. Use a Jupyter kernel with **GPU PyTorch** (cell 1 checks).
3. No internet needed — encoders load offline from the zip.

## 1 · GPU / env check

In [ ]:
import os
!nvidia-smi
import torch
print('\npython :', os.sys.executable)
print('torch  :', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not visible — switch to a CUDA-enabled kernel.'

## 2 · Config — edit these
`EPOCHS=1` is a smoke test; `100` is the real recipe (early stopping usually ends sooner).
`WORKERS=0` avoids the data-loader multiprocessing hang we hit on the server — raise it only
if you confirm the loader is stable on ORC. Resume is automatic; re-run the loop cell after any kill.

In [ ]:
import os
USER = os.environ.get('USER', os.environ.get('LOGNAME','user')); SCRATCH = f'/scratch/{USER}'
ZIP_PATH = f'{SCRATCH}/orc_bundle.zip'    # copy the bundle here first
WORK     = f'{SCRATCH}/orc_bundle'        # unzip target (persistent /scratch)

EPOCHS  = 100      # 1 = smoke test
WORKERS = 0        # 0 = safe (no loader hang). raise only if stable.
SEED    = 42       # student training seed (fixed across arms)
ALPHA   = 0.75     # fallback KD mix if the search is turned off
DEVICE  = 'cuda:0'
# --- alpha search on VALIDATION (grid; Optuna auto-used if installed) ---
RUN_ALPHA_SEARCH   = True   # search the KD mix on val before the full runs
ALPHA_TRIALS       = 5      # grid points / optuna trials
ALPHA_SEARCH_EPOCHS= 15     # short runs for the search (full runs use EPOCHS)
print('USER', USER, '| WORK', WORK, '| EPOCHS', EPOCHS, '| WORKERS', WORKERS,
      '| alpha search', RUN_ALPHA_SEARCH)

## 3 · Unzip the bundle (persistent /scratch) + cd into it

In [ ]:
import zipfile, time
assert os.path.exists(ZIP_PATH), f'copy orc_bundle.zip to {SCRATCH}/ first (not found at {ZIP_PATH})'
if not os.path.isdir(WORK) or not os.path.exists(f'{WORK}/kd_core.py'):
    os.makedirs(WORK, exist_ok=True); t=time.time()
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(WORK)
    print(f'unzipped in {time.time()-t:.0f}s')
else:
    print('already unzipped:', WORK)
os.chdir(WORK); print('cwd =', os.getcwd())

## 4 · Deps + offline / cuDNN env fixes

In [ ]:
import importlib, subprocess, sys
def _have(m):
    try: importlib.import_module(m); return True
    except Exception: return False
miss = [m for m in ('transformers','albumentations','cv2','pandas','tqdm','scipy') if not _have(m)]
if miss:
    print('installing', miss); subprocess.call([sys.executable,'-m','pip','install','--user','-q',
        *[{'cv2':'opencv-python-headless'}.get(m,m) for m in miss]])
import transformers; print('transformers', transformers.__version__)
# offline HF (encoders load from the zip) + let torch use its bundled cuDNN
os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['LD_LIBRARY_PATH']=':'.join(p for p in os.environ.get('LD_LIBRARY_PATH','').split(':')
    if p and 'cuda' not in p.lower() and 'cudnn' not in p.lower())

## 5 · Verify the bundle has everything (teachers, encoders, data, references)

In [ ]:
need = ['kd_core.py','distill_segformer.py','aggregate_report.py','val_oracle.py',
        'teachers/segformer_b5_teacher/best_model.pt','teachers/segformer_b5_teacher/temperature.json',
        'teachers/segformer_b2_teacher/best_model.pt','teachers/segformer_b2_teacher/temperature.json',
        'pretrained_weights/segformer_mit_b0/config.json','pretrained_weights/segformer_mit_b2/config.json',
        'pretrained_weights/segformer_mit_b5/config.json','manifests/train_manifest.csv',
        'manifests/test_manifest.csv','ita_labels/wl_test_per_image_ita.csv',
        'ita_labels/wl_train_per_image_ita.csv',
        'distill_out/reference/segformer_b0_distilled_test_per_image.csv']
ok=True
for p in need:
    e=os.path.exists(p); ok = ok and e; print('OK  ' if e else 'MISS', p)
assert ok, 'bundle is missing files above — rebuild orc_bundle.zip with make_orc_bundle.py'
print('\nall required files present.')

## 6 · Validation oracle result (from the bundle) — the multi-teacher gate
This was computed on the server on **validation only**. It tells us whether B2 & B5 are
complementary (so combining them helps) and gives `rel_b` for the reliability gate.

In [ ]:
import json
vo = 'distill_out/val_oracle/val_oracle.json'
if os.path.exists(vo):
    d = json.load(open(vo)); print(json.dumps(d, indent=2))
    REL_B = float(d['rel_b_for_adaptive_gate'])
    print(f"\nGATE run adaptive = {d['GATE_run_adaptive']} | oracle gain {d['oracle_gain_over_best_single']:+.4f} "
          f"CI {d['oracle_gain_ci95']} | rel_b = {REL_B:.3f}")
else:
    REL_B = 0.5; print('no val_oracle.json in bundle -> using rel_b = 0.5 (rerun val_oracle if needed)')

## 7 · Define the experiment arms
Each is one distilled B0 student. Paths point at the bundled teachers/encoders. This is the
exact set the server runs — response / CWD / BPKD / uniform ensemble / the reliability-gated
novel method + its four ablations / angular / DKD.

In [ ]:
T_B5='teachers/segformer_b5_teacher'; P_B5='pretrained_weights/segformer_mit_b5'
T_B2='teachers/segformer_b2_teacher'; P_B2='pretrained_weights/segformer_mit_b2'
P_B0='pretrained_weights/segformer_mit_b0'
ITA_TEST='ita_labels/wl_test_per_image_ita.csv'; ITA_TRAIN='ita_labels/wl_train_per_image_ita.csv'
OUT='distill_out'
COMMON=(f'--train-manifest manifests/train_manifest.csv --test-manifest manifests/test_manifest.csv '
        f'--data-root . --student-pretrained {P_B0} --test-ita {ITA_TEST} '
        f'--seed {SEED} --workers {WORKERS} --epochs {EPOCHS} --device {DEVICE}')
ADAPT=(f'--teacher-a {T_B2} --pretrained-a {P_B2} --teacher-b {T_B5} --pretrained-b {P_B5} '
       f'--ensemble adaptive --rel-b {REL_B} --kd response')
ARMS = [
  ('expA_b5_to_b0_response', f'--teacher-a {T_B5} --pretrained-a {P_B5} --kd response'),
  ('p2_cwd_b5_to_b0',        f'--teacher-a {T_B5} --pretrained-a {P_B5} --kd cwd'),
  ('p2_bpkd_b5_to_b0',       f'--teacher-a {T_B5} --pretrained-a {P_B5} --kd bpkd'),
  ('p2_ensemble_uniform',    f'--teacher-a {T_B2} --pretrained-a {P_B2} --teacher-b {T_B5} --pretrained-b {P_B5} --ensemble uniform --kd response'),
  ('p3_adaptive',            ADAPT),
  ('p3_adaptive_boundary',   ADAPT + ' --boundary'),
  ('p3_adaptive_hard',       ADAPT + ' --hard'),
  ('p3_adaptive_group',      ADAPT + f' --group --train-ita {ITA_TRAIN}'),
  ('p3_adaptive_full',       ADAPT + f' --boundary --hard --group --train-ita {ITA_TRAIN}'),
  ('x_angular_b5_to_b0',     f'--teacher-a {T_B5} --pretrained-a {P_B5} --kd angular'),
  ('x_dkd_b5_to_b0',         f'--teacher-a {T_B5} --pretrained-a {P_B5} --kd dkd'),
]
print(f'{len(ARMS)} arms:'); [print('  ', r) for r,_ in ARMS]

## 8 · Search the KD mix (alpha) on VALIDATION
Short reduced-epoch runs at several alpha values; keeps the one with the best **val** Dice.
Uses a grid by default (Optuna auto-used if it's installed). One alpha for the single-teacher
family (response/CWD/BPKD/angular/DKD) and one for the ensemble family (uniform/adaptive/
ablations). Cached to `distill_out/optuna_alpha/` so re-running is instant. Set
`RUN_ALPHA_SEARCH=False` to skip and use the fixed `ALPHA`.

In [ ]:
import json
ALPHA_SINGLE = ALPHA; ALPHA_ENS = ALPHA
def search_alpha(tag, teach_args):
    js = f'{OUT}/optuna_alpha/{tag}_best_alpha.json'
    if not os.path.exists(js):
        cmd = (f'python optuna_alpha.py --tag {tag} --n-trials {ALPHA_TRIALS} '
               f'--search-epochs {ALPHA_SEARCH_EPOCHS} --out-dir {OUT} -- {COMMON} {teach_args}')
        print('\n' + '='*72 + f'\nALPHA SEARCH  {tag}\n' + '='*72)
        get_ipython().system(cmd)
    return json.load(open(js))['best_alpha'] if os.path.exists(js) else ALPHA
if RUN_ALPHA_SEARCH:
    ALPHA_SINGLE = search_alpha('single_b5_response',
        f'--teacher-a {T_B5} --pretrained-a {P_B5} --kd response')
    ALPHA_ENS = search_alpha('ensemble_uniform',
        f'--teacher-a {T_B2} --pretrained-a {P_B2} --teacher-b {T_B5} --pretrained-b {P_B5} --ensemble uniform --kd response')
print(f'\nALPHA_SINGLE = {ALPHA_SINGLE:.3f}   ALPHA_ENS = {ALPHA_ENS:.3f}')

## 9 · Run all pending arms (resumable)
Loops over the arms: **skips** any with a `DONE.json`, **resumes** an interrupted one, trains
the rest — each at the searched alpha for its family. Re-run this cell after a wall-time kill —
it continues where it left off. Each arm prints per-epoch `val_dice`, then a final
`TEST dice=… miss=… rec<.1=… p5subj=… worst-grp=…` line.

In [ ]:
import os
for run_id, extra in ARMS:
    if os.path.exists(f'{OUT}/{run_id}/DONE.json'):
        print(f'[skip done] {run_id}'); continue
    a = ALPHA_ENS if '--ensemble' in extra else ALPHA_SINGLE
    cmd = f'python -u distill_segformer.py --run-id {run_id} --out-dir {OUT} {COMMON} --alpha {a} {extra}'
    print('\n' + '='*72 + f'\nRUN  {run_id}   (alpha={a:.3f})\n' + '='*72)
    get_ipython().system(cmd)
print('\n[all arms attempted] re-run this cell if any were interrupted.')

## 10 · Score every finished arm vs the B2→B0 reference (WIN / NON-INFERIOR / INFERIOR)

In [ ]:
REF = 'distill_out/reference/segformer_b0_distilled_test_per_image.csv'
arms_str = ' '.join(f'{r}={OUT}/{r}/test_per_image.csv'
                    for r,_ in ARMS if os.path.exists(f'{OUT}/{r}/test_per_image.csv'))
if arms_str:
    get_ipython().system(f'python aggregate_report.py --ref {REF} --ita {ITA_TEST} --arms {arms_str} --reps 5000 --ni-margin 0.03 --out {OUT}/aggregate')
else:
    print('no finished arms yet — run cell 8 first.')

## 11 · Results table (everything scored so far, incl. teachers)

In [ ]:
import glob, json, numpy as np, pandas as pd
paths = sorted(set(glob.glob('distill_out/**/test_per_image.csv', recursive=True)
                 + glob.glob('teachers/**/test_per_image.csv', recursive=True)))
rows=[]
for p in paths:
    name = os.path.basename(os.path.dirname(p)); d = pd.read_csv(p)
    miss = ((d.pred_positive_pixels==0)&(d.gt_positive_pixels>0))
    r = {'model':name,'n':len(d),'mean_dice':round(d.dice.mean(),4),'median_dice':round(d.dice.median(),4),
         'miss%':round(miss.mean()*100,2),'rec<.10%':round((d.recall<0.10).mean()*100,2),
         'dice<.20%':round((d.dice<0.20).mean()*100,2)}
    if 'subject' in d.columns:
        r['p5_subj']=round(np.percentile(d.groupby('subject').dice.mean(),5),4)
    rows.append(r)
if rows:
    df=pd.DataFrame(rows).sort_values('median_dice',ascending=False)
    pd.set_option('display.width',200,'display.max_columns',20)
    print(df.to_string(index=False))
else:
    print('nothing scored yet.')

## 12 · (optional) See what a model predicts — visual sanity check
Loads one finished student + the B5 teacher and overlays their predictions on a few test
images next to the ground truth, so you can *see* the difference. Edit `WHICH` to any run_id.

In [ ]:
WHICH = 'p3_adaptive'   # any finished arm; falls back to a teacher if missing
import numpy as np, pandas as pd, torch, cv2, importlib.util, json
spec = importlib.util.spec_from_file_location('kd_core','kd_core.py')
K = importlib.util.module_from_spec(spec); spec.loader.exec_module(K)
dev = torch.device(DEVICE)
def load_model(pre, ckpt):
    m=K.SegformerWrapper(K.build_segformer(pre,num_labels=1)).to(dev)
    m.load_state_dict(torch.load(ckpt,map_location=dev,weights_only=True)); m.eval(); return m
ck = f'{OUT}/{WHICH}/best_model.pt'
if os.path.exists(ck): student=load_model(P_B0, ck); title=WHICH
else: student=load_model(P_B5, f'{T_B5}/best_model.pt'); title='segformer_b5_teacher'
thr = 0.5
tj = f'{OUT}/{WHICH}/DONE.json'
if os.path.exists(tj): thr = json.load(open(tj)).get('threshold',0.5)
test = K.load_manifest('manifests/test_manifest.csv','.')
import albumentations as A; from albumentations.pytorch import ToTensorV2
tf = A.Compose([A.Resize(640,640),A.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),ToTensorV2()])
try:
    import matplotlib.pyplot as plt
    rows = test.sample(3, random_state=0).reset_index(drop=True)
    fig,ax=plt.subplots(len(rows),3,figsize=(11,3.4*len(rows)))
    for i,r in rows.iterrows():
        im=cv2.cvtColor(cv2.imread(str(r.image_path)),cv2.COLOR_BGR2RGB)
        gt=cv2.imread(str(r.mask_path),cv2.IMREAD_GRAYSCALE)
        if gt.ndim==3: gt=gt[...,0]
        x=tf(image=im,mask=(gt>0).astype('float32'))['image'].unsqueeze(0).float().to(dev)
        with torch.no_grad(), torch.amp.autocast('cuda'): pr=torch.sigmoid(student(x).float())[0,0].cpu().numpy()
        pm=(cv2.resize(pr,(im.shape[1],im.shape[0]))>=thr).astype('uint8')
        ax[i,0].imshow(im); ax[i,0].set_title('image'); ax[i,0].axis('off')
        ax[i,1].imshow(im); ax[i,1].imshow(gt>0,alpha=0.4,cmap='Greens'); ax[i,1].set_title('ground truth'); ax[i,1].axis('off')
        ax[i,2].imshow(im); ax[i,2].imshow(pm,alpha=0.4,cmap='Reds'); ax[i,2].set_title(f'{title} pred'); ax[i,2].axis('off')
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed — skip the visual.')

## Done
- **Interrupted?** Just re-run cell 9 — it skips finished arms and resumes the rest.
- **Results:** cell 10 = verdicts vs the B0-distilled baseline; cell 11 = raw metrics table.
- All outputs are on persistent `/scratch/$USER/orc_bundle/distill_out/` — nothing to copy back.